## Reuters 뉴스 카테고리 분류 - 모델별 비교 프로젝트


## 회고
- 가장 좋았던 모델 : Linear SVC (0.8322)
- 두 번째로 좋았던 모델 : LogisticRegression (0.8274)
- 회고: 딥러닝 F1-score가 0.8을 넘기지 못했다. 텍스트에는 선형 모델이 강하다는 것을 알 수 있었다.


### 📊 단어장 개수별 ML 모델 성능 비교 (Accuracy / F1-score)

| #Words| Model              | Accuracy | F1-score |
|-----------|--------------------|----------|----------|
| 5,000     | LogisticRegression | 0.8250   | 0.8274   |
|           | SVM                | 0.8050   | 0.7965   |
|           | Linear SVC ⭐       | 0.8388   | 0.8322   |
|           | RandomForest       | 0.7863   | 0.7787   |
|           | ComplementNB       | 0.7707   | 0.7459   |
|           | XGBoost            | 0.8241   | 0.8171   |
|           | LightGBM           | 0.8072   | 0.7979   |
|           | Dense              | 0.8045   | 0.7949   |


In [ ]:
from tensorflow.keras.datasets import reuters
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
from tensorflow.keras.layers import Input, Dense, Dropout, Activation
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers

In [23]:
import warnings
warnings.filterwarnings('ignore')

## 데이터 준비
- index -> text
- TF-idf 학습 데이터 준비

In [4]:
# load reuters data

(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=5000, test_split=0.2)

In [5]:
word_index = reuters.get_word_index(path="reuters_word_index.json")

In [6]:
index_to_word = { index+3 : word for word, index in word_index.items() }
for index, token in enumerate(("<pad>", "<sos>", "<unk>")):
  index_to_word[index]=token

In [7]:
decoded = []
for i in range(len(x_train)):
    t = ' '.join([index_to_word[index] for index in x_train[i]])
    decoded.append(t)

x_train = decoded
print(len(x_train))

8982


In [8]:
decoded_test = []
for i in range(len(x_test)):
    t = ' '.join([index_to_word[index] for index in x_test[i]])
    decoded_test.append(t)

x_test = decoded_test
print(len(x_test))

2246


In [9]:
# 벡터화 TF-idf
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer() 

x_train_tfidf = tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf = tfidf_vectorizer.transform(x_test)

In [10]:
x_train_tfidf.shape

(8982, 4867)

In [11]:
x_test_tfidf.shape

(2246, 4867)

In [12]:
x_train[3]

"<sos> the farmers home administration the u s agriculture department's farm lending arm could lose about seven billion dlrs in outstanding principal on its severely <unk> borrowers or about one fourth of its farm loan portfolio the general accounting office gao said in remarks prepared for delivery to the senate agriculture committee brian <unk> senior associate director of gao also said that a preliminary analysis of proposed changes in <unk> financial <unk> standards indicated as many as one half of <unk> borrowers who received new loans from the agency in 1986 would be <unk> under the proposed system the agency has proposed <unk> <unk> credit using a variety of financial <unk> instead of <unk> <unk> on <unk> ability senate agriculture committee chairman <unk> <unk> d <unk> <unk> the proposed <unk> changes telling <unk> administrator <unk> clark at a hearing that they would mark a dramatic shift in the <unk> purpose away from being <unk> <unk> of last <unk> toward becoming a big cit

## 모델 정의 및 실험
- Logistic Regression
- SVM
- Linear SVC
- XGBoost
- LightGBM
- Random Forest
- ComplementNB
- dense
- voting

In [19]:
# 머신러닝 모델 정의

models = {'Logistic Regression': LogisticRegression(C=10, penalty='l2', max_iter=3000, class_weight='balanced', n_jobs=-1),
          'XGBoost': XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=6, 
                                   colsample_bytree=0.8, subsample=0.8,
                                   eval_metric='mlogloss', tree_method='hist', device='cuda', random_state=0, verbosity=0),
          'LGBM': LGBMClassifier(n_estimators=500, learning_rate=0.1, max_depth=-1, 
                                 colsample_bytree=0.8, subsample=0.8, n_jobs=-1, random_state=0, verbose=-1),
        'SVM': SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced', random_state=0),
          'Linear SVC': LinearSVC(C=10, penalty='l1', dual=False, max_iter=3000, class_weight='balanced', random_state=0),
          'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=None, 
                                                 class_weight='balanced', n_jobs=-1, random_state=0),
          'ComplementNB': ComplementNB(alpha=1.0)}

In [ ]:
# 모델 학습
results =[]

for name, model in models.items():
    model.fit(x_train_tfidf, y_train)
    predicted = model.predict(x_test_tfidf)

    acc = round(accuracy_score(y_test, predicted), 4)
    f1 = round(f1_score(y_test, predicted, average='weighted'), 4)
    print(f'{name} 학습 완료! 정확도: {acc}, F1-score: {f1}')
    
    results.append({
                'Name': name,
                'Accuracy Score': acc,
                'F1 Score': f1
                })

results_df = pd.DataFrame(results)
print('Vocab Size: 5000 실험 결과')
print(results_df)

Logistic Regression 학습 완료! 정확도: 0.825, F1-score: 0.8274
XGBoost 학습 완료! 정확도: 0.8112, F1-score: 0.8047
LGBM 학습 완료! 정확도: 0.3557, F1-score: 0.3629
SVM 학습 완료! 정확도: 0.805, F1-score: 0.7965
Linear SVC 학습 완료! 정확도: 0.7979, F1-score: 0.7983
RandomForest 학습 완료! 정확도: 0.7863, F1-score: 0.7787
ComplementNB 학습 완료! 정확도: 0.7707, F1-score: 0.7459
Vocab Size:5000 실험 결과
                  Name  Accuracy Score  F1 Score
0  Logistic Regression          0.8250    0.8274
1              XGBoost          0.8112    0.8047
2                 LGBM          0.3557    0.3629
3                  SVM          0.8050    0.7965
4           Linear SVC          0.7979    0.7983
5         RandomForest          0.7863    0.7787
6         ComplementNB          0.7707    0.7459


In [24]:
# LGBM 재학습
lgbm = LGBMClassifier(n_estimators=1000, learning_rate=0.01, max_depth=6, 
                                 colsample_bytree=0.8, subsample=0.8, n_jobs=-1, random_state=0, verbose=-1)
lgbm.fit(x_train_tfidf, y_train)
predicted = lgbm.predict(x_test_tfidf)
acc = round(accuracy_score(y_test, predicted), 4)
f1 = round(f1_score(y_test, predicted, average='weighted'), 4)
print(f'LGBM 학습 완료! 정확도: {acc}, F1-score: {f1}')

LGBM 학습 완료! 정확도: 0.8072, F1-score: 0.7979


In [27]:
# XGBoost 재학습
xgb = XGBClassifier(n_estimators=500, learning_rate=0.01, max_depth=6, 
                                   colsample_bytree=0.8, subsample=0.8,
                                   eval_metric='mlogloss', tree_method='hist', device='cuda', random_state=0, verbosity=0)
xgb.fit(x_train_tfidf, y_train)
predicted = xgb.predict(x_test_tfidf)
acc = round(accuracy_score(y_test, predicted), 4)
f1 = round(f1_score(y_test, predicted, average='weighted'), 4)
print(f'XGBoost 학습 완료! 정확도: {acc}, F1-score: {f1}')

XGBoost 학습 완료! 정확도: 0.8241, F1-score: 0.8171


In [ ]:
# Linear SVC 재학습

# 모델 정의
lsvc = LinearSVC(penalty='l1', dual=False, C=1.0, max_iter=3000)

# 학습
lsvc.fit(x_train_tfidf, y_train)

# 예측
predicted = lsvc.predict(x_test_tfidf)

# 평가
c_matrix = confusion_matrix(y_test, predicted)
print(c_matrix)
print(f'Accuracy score: {accuracy_score(y_test, predicted)}')
print(f'F1 score: {f1_score(y_test, predicted, average='weighted')}')

[[ 8  1  0 ...  0  0  0]
 [ 0 87  1 ...  0  0  0]
 [ 0  0 15 ...  0  0  0]
 ...
 [ 0  0  0 ...  6  0  0]
 [ 0  1  0 ...  0  4  0]
 [ 0  0  0 ...  0  0  0]]
Accuracy score: 0.8388245770258237
F1 score: 0.8322057512153768


In [86]:
from sklearn.model_selection import train_test_split

seed = 42
num_words = 5000

tf.keras.backend.clear_session()
tf.random.set_seed(seed)
np.random.seed(seed)

# Load + split
(x_train, y_train), (x_test, y_test) = reuters.load_data(num_words=num_words, test_split=0.2)

y_train = np.array(y_train)
y_test = np.array(y_test)

x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train,
    test_size=0.2,
    random_state=seed,
    stratify=y_train
)

# Decode
word_index = reuters.get_word_index()
index_to_word = {v + 3: k for k, v in word_index.items()}
index_to_word[0] = '<pad>'
index_to_word[1] = '<start>'
index_to_word[2] = '<unk>'

# <pad> <start> <unk> 제거
def seqs_to_texts(seqs):
    return [' '.join(index_to_word.get(i, '<unk>') for i in seq if i > 2) for seq in seqs]

x_train_text = seqs_to_texts(x_train)
x_val_text = seqs_to_texts(x_val)
x_test_text = seqs_to_texts(x_test)

# TF-IDF
tfidf = TfidfVectorizer()
x_train_tfidf = tfidf.fit_transform(x_train_text)
x_val_tfidf = tfidf.transform(x_val_text)
x_test_tfidf = tfidf.transform(x_test_text)

n_features = x_train_tfidf.shape[1]

In [88]:
print('x_val_tfidf shape:', x_val_tfidf.shape)
print('x_test_tfidf shape:', x_test_tfidf.shape)

x_val_tfidf shape: (1797, 4864)
x_test_tfidf shape: (2246, 4864)


In [115]:
# Model
l2_lambda = 1e-3

inputs = Input(shape=(n_features,))
x = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(l2_lambda))(inputs)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu', kernel_regularizer=regularizers.l2(l2_lambda))(x)
x = Dropout(0.3)(x)
outputs = Dense(46, activation='softmax')(x)

model = Model(inputs=inputs, outputs=outputs)


In [116]:
# Callbacks

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)
lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)

In [117]:
# Complie
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Fit
history = model.fit(
    x_train_tfidf.toarray(),
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(x_val_tfidf.toarray(), y_val),
    callbacks=[early_stop, lr_reducer]
)

Epoch 1/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.5857 - loss: 2.0970 - val_accuracy: 0.7090 - val_loss: 1.5451 - learning_rate: 0.0010
Epoch 2/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7180 - loss: 1.4814 - val_accuracy: 0.7340 - val_loss: 1.4005 - learning_rate: 0.0010
Epoch 3/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7587 - loss: 1.3485 - val_accuracy: 0.7646 - val_loss: 1.3374 - learning_rate: 0.0010
Epoch 4/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7826 - loss: 1.2725 - val_accuracy: 0.7724 - val_loss: 1.3035 - learning_rate: 0.0010
Epoch 5/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7943 - loss: 1.2212 - val_accuracy: 0.7807 - val_loss: 1.2814 - learning_rate: 0.0010
Epoch 6/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8046 - loss: 1.1831 - val_accuracy: 0.7896 - val_loss: 1.2671 - learning_rate: 0.0010
Epoch 7/100
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8122 - loss: 1.

In [118]:
# softmax 확률 (N, 46)
proba = model.predict(x_test_tfidf.toarray(), batch_size=1024, verbose=0)

# 예측 라벨 (N,)
pred = np.argmax(proba, axis=1)

acc = round(accuracy_score(y_test, pred), 4)
f1 = round(f1_score(y_test, pred, average='weighted'), 4)

print(f'DL 정확도: {acc}, F1-score: {f1}')

DL 정확도: 0.805, F1-score: 0.794
